# Modelo 4 — Análisis final de experimentos

**Qué es el Modelo 4.** MILP que asigna cursos de dos bloques consecutivos a
salas, minimizando el costo total de movimiento estudiantil
`sum f_ik * c_rs * z_ikrs`, con `z_ikrs` linealizando `x_ir * y_ks`
(McCormick). `x`, `y` binarias; `z` continua en [0,1].

**Matriz detallada sala–sala.** `c_rs` se construye con distancia euclidiana
3D entre salas (coord_z = piso × altura) más una penalización por tipo de
transición: misma sala (0), mismo piso, distinto piso, cambio de edificio.

**Edificios y pisos.** Cada sala pertenece a un edificio y piso, con
coordenadas físicas; esto hace que moverse dentro del edificio sea barato y
cambiar de edificio sea caro.

**Carreras y años en los flujos.** Los flujos `f_ik` se generan con pesos:
altos entre cursos de la misma carrera y año, medios entre años de la misma
carrera, bajos entre carreras (electivos). Además hay **estudiantes libres**
(salen tras el bloque 1) y **estudiantes entrantes** (no tuvieron clases en
el bloque 1 pero sí en el 2). Ninguno genera costo, pero ambos afectan los
tamaños de los cursos y, por tanto, las restricciones de capacidad:
`sum_k f_ik + libres_i = tamano_i` y `sum_i f_ik + entrantes_k = tamano_k`.

**Análisis de complejidad.** Cómo crece el modelo (variables `z`,
restricciones de linealización) con el tamaño de la instancia.

**Análisis de escalabilidad.** Cómo responde el solver (tiempo, gap) al
aumentar I, K y R (Familia A).

**Gap de optimalidad.** `(obj − best_bound) / obj`: distancia relativa entre
la mejor solución encontrada y la mejor cota inferior. Gap 0 = óptimo
certificado; gap > 0 con TIME_LIMIT = solución factible sin certificado.

**Mejora frente al azar.** `(costo_azar_promedio − obj) / costo_azar_promedio × 100`,
con 10 asignaciones aleatorias factibles como línea base.

**Sobreoferta de salas.** Familia B: con I=K=18 fijos se aumenta R
(18→36) para medir el efecto de capacidades/salas excedentes sobre el costo,
el tiempo de resolución y las salas sin usar.


## Carga de resultados reales
El pipeline se ejecuta desde terminal con:
```bash
python src/correr_experimentos_m4.py
python src/construir_notebook_final.py
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd()
if not (BASE / "outputs").exists():     # permite ejecutar desde notebooks/
    BASE = BASE.parent
DIR_RES = BASE / "outputs" / "resultados"
DIR_GRAF = BASE / "outputs" / "graficos"
DIR_GRAF.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DIR_RES / "resultados_modelo4.csv")
fa = df[df["familia"] == "A_escalabilidad"].sort_values("I").copy()
fb = df[df["familia"] == "B_sobreoferta"].sort_values("R").copy()
df


## Tabla resumen

In [ ]:
cols = ["instancia", "familia", "I", "K", "R", "num_z", "num_constraints",
        "status", "obj_modelo", "best_bound", "mip_gap", "runtime",
        "mejora_vs_azar", "total_libres", "total_entrantes"]
tabla = df[[c for c in cols if c in df.columns]]
sin_solucion = tabla[tabla["obj_modelo"].isna()]
if len(sin_solucion):
    print("Instancias SIN solución factible reportada:",
          sin_solucion["instancia"].tolist())
tabla


## Escalabilidad: tiempo vs tamaño

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fa["I"], fa["runtime"], "o-")
for _, f in fa.iterrows():
    ax.annotate(f["status"], (f["I"], f["runtime"]), fontsize=8,
                textcoords="offset points", xytext=(4, 4))
ax.set_xlabel("Tamaño (I = K = R)"); ax.set_ylabel("Tiempo [s]")
ax.set_title("Familia A: tiempo de resolución vs tamaño")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_GRAF / "A_tiempo_vs_tamano.png", dpi=150)
plt.show()

## Escalabilidad: gap vs tamaño

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fa["I"], 100 * fa["mip_gap"].fillna(np.nan), "s-", color="tab:red")
ax.set_xlabel("Tamaño (I = K = R)"); ax.set_ylabel("Gap de optimalidad [%]")
ax.set_title("Familia A: gap vs tamaño")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_GRAF / "A_gap_vs_tamano.png", dpi=150)
plt.show()

## Complejidad: num_z vs runtime

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["num_z"], df["runtime"], c=(df["familia"] == "B_sobreoferta"),
           cmap="coolwarm", s=60)
for _, f in df.iterrows():
    ax.annotate(f["instancia"], (f["num_z"], f["runtime"]), fontsize=7,
                textcoords="offset points", xytext=(4, 2))
ax.set_xlabel("Número de variables z creadas (f_ik > 0)")
ax.set_ylabel("Tiempo [s]")
ax.set_title("Complejidad: num_z vs runtime (todas las instancias)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_GRAF / "numz_vs_runtime.png", dpi=150)
plt.show()

## Mejora frente al azar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fa["instancia"], fa["mejora_vs_azar"], color="tab:green")
ax.set_ylabel("Mejora vs azar [%]")
ax.set_title("Familia A: mejora del Modelo 4 frente a asignaciones aleatorias")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(DIR_GRAF / "A_mejora_vs_azar.png", dpi=150)
plt.show()

## Sobreoferta: costo vs número de salas

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fb["R"], fb["obj_modelo"], "o-", label="Costo Modelo 4")
ax.plot(fb["R"], fb["costo_azar_promedio"], "s--", color="gray",
        label="Costo azar (promedio)")
ax.set_xlabel("Número de salas R (I = K = 18)")
ax.set_ylabel("Costo objetivo")
ax.set_title("Familia B: efecto de la sobreoferta de salas en el costo")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_GRAF / "B_costo_vs_salas.png", dpi=150)
plt.show()

## Sobreoferta: runtime y gap

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(fb["R"], fb["runtime"], "o-")
ax1.set_xlabel("R"); ax1.set_ylabel("Tiempo [s]")
ax1.set_title("Familia B: runtime vs sobreoferta"); ax1.grid(alpha=0.3)
ax2.plot(fb["R"], 100 * fb["mip_gap"].fillna(np.nan), "s-", color="tab:red")
ax2.set_xlabel("R"); ax2.set_ylabel("Gap [%]")
ax2.set_title("Familia B: gap vs sobreoferta"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_GRAF / "B_runtime_gap.png", dpi=150)
plt.show()

## Estudiantes por tipo de transición

In [ ]:
d = df.dropna(subset=["obj_modelo"])
xpos = np.arange(len(d)); w = 0.27
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(xpos - w, d["estudiantes_misma_sala"], w, label="Misma sala")
ax.bar(xpos,     d["estudiantes_mismo_edificio"], w, label="Mismo edificio")
ax.bar(xpos + w, d["estudiantes_cambio_edificio"], w, label="Cambio edificio")
ax.set_xticks(xpos); ax.set_xticklabels(d["instancia"], rotation=45, ha="right")
ax.set_ylabel("Estudiantes")
ax.set_title("Estudiantes por tipo de transición (solución del Modelo 4)\n"
             "Nota: libres y entrantes no se mueven y no aparecen aquí")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(DIR_GRAF / "estudiantes_por_transicion.png", dpi=150)
plt.show()

## Asignación representativa

In [ ]:
# Asignación representativa: instancia factible más grande (mayor I + R).
d = df.dropna(subset=["obj_modelo"]).copy()
if len(d):
    d["tam_total"] = d["I"] + d["R"]
    nombre = d.sort_values("tam_total").iloc[-1]["instancia"]
    print(f"Instancia representativa: {nombre}")
    a1 = pd.read_csv(BASE / "outputs" / "asignaciones" / nombre /
                     "asignaciones_b1.csv")
    a2 = pd.read_csv(BASE / "outputs" / "asignaciones" / nombre /
                     "asignaciones_b2.csv")
    display(a1.head(15)); display(a2.head(15))
else:
    print("Ninguna instancia tiene solución factible reportada.")


## Conclusiones preliminares

Lea estas conclusiones junto con la tabla real de resultados de arriba
(se generan a partir de `resultados_modelo4.csv`, no de valores inventados):

- **Complejidad:** `num_z` y `num_constraints` crecen rápidamente con I·K·R²;
  compare `num_z` contra `runtime` en el gráfico correspondiente.
- **Escalabilidad:** observe en la Familia A desde qué tamaño el solver deja
  de certificar óptimo dentro del `time_limit` (status TIME_LIMIT, gap > 0).
- **Mejora vs azar:** la optimización debería reducir fuertemente el costo
  frente a asignaciones aleatorias factibles; verifique la magnitud real.
- **Sobreoferta:** más salas tienden a reducir el costo óptimo (más
  flexibilidad), a costa de un espacio de búsqueda mayor; observe el
  trade-off en runtime/gap y las salas no usadas.
- **Libres y entrantes:** no aparecen en la función objetivo, pero al inflar
  los tamaños de los cursos pueden forzar cambios de sala vía capacidad.
- Las instancias sin óptimo certificado se reportan como **factibles con gap
  positivo**; si alguna quedara sin solución factible, aparece indicada en la
  tabla con `obj_modelo` vacío.
